# Descarga de Datos

Se descargan las tablas desde neon.tech y se almacenan en formato `parquet` para reducir el uso de la conexión. De ser necesario actualizar los archivos, cambiar el parametro refresh a **True**

In [15]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent


sys.path.insert(0, str(RAIZ))

In [16]:
import numpy as np

In [17]:
from src.db.connection import get_conn

In [18]:
import pandas as pd
conn = get_conn()

def get_tables(refresh = False):
    if refresh:

        with conn.cursor() as cursor:
            cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'core'")
            return [row[0] for row in cursor.fetchall()]


In [19]:
tablas = get_tables(refresh=False)
print(tablas)

None


In [20]:
from pathlib import Path

out = Path(RAIZ/"src/data/parquet")
out.mkdir(exist_ok=True)

def save_parquet(table_name, refresh = False):
    if refresh: 
        for t in tablas:
            df = pd.read_sql(f'SELECT * FROM core."{t}"', conn)
            df.to_parquet(out / f"{t}.parquet", engine="pyarrow", index=False)
            print(f"{t}: {len(df):,} filas")

In [21]:
save_parquet(tablas, refresh=False)

In [22]:
#cerrar conexion
conn.close()

In [23]:
df_shots = pd.read_parquet(RAIZ/"src/data/parquet/fct_shots.parquet")
df_shots.isnull().sum()

game_gevent_id         0
game_team_id           0
game_id                0
game_event_id          0
season_num             0
game_date              0
team_id                0
team_name              0
player_id              0
player_name            0
htm                    0
vtm                    0
period                 0
minutes_remaining      0
seconds_remaining      0
grid_type              0
event_type             0
action_type            0
shot_type              0
shot_zone_basic        0
shot_zone_area         0
shot_zone_range        0
shot_distance          0
loc_x                  0
loc_y                  0
shot_attempted_flag    0
shot_made_flag         0
zone_id                0
game_player_id         0
dtype: int64

In [24]:
df_shots.head(5)

,game_gevent_id,game_team_id,game_id,game_event_id,season_num,game_date,team_id,team_name,player_id,player_name,...,shot_zone_basic,shot_zone_area,shot_zone_range,shot_distance,loc_x,loc_y,shot_attempted_flag,shot_made_flag,zone_id,game_player_id
0,0022400006-9,0022400006-1610612762,0022400006,9,2024-25,2024-11-12,1610612762,Utah Jazz,1628381,John Collins,...,Mid-Range,Right Side Center(RC),16-24 ft.,17,-108,-138,1,0,LMR_R,0022400006-1628381
1,0022400006-12,0022400006-1610612762,0022400006,12,2024-25,2024-11-12,1610612762,Utah Jazz,1628381,John Collins,...,In The Paint (Non-RA),Right Side(R),8-16 ft.,10,-76,-75,1,1,P_HIGH,0022400006-1628381
2,0022400006-16,0022400006-1610612762,0022400006,16,2024-25,2024-11-12,1610612762,Utah Jazz,1628374,Lauri Markkanen,...,Above the Break 3,Left Side Center(LC),24+ ft.,24,132,-201,1,0,ATB3_L,0022400006-1628374
3,0022400006-22,0022400006-1610612762,0022400006,22,2024-25,2024-11-12,1610612762,Utah Jazz,1629012,Collin Sexton,...,In The Paint (Non-RA),Center(C),8-16 ft.,9,-34,-88,1,0,P_HIGH,0022400006-1629012
4,0022400006-28,0022400006-1610612762,0022400006,28,2024-25,2024-11-12,1610612762,Utah Jazz,1628374,Lauri Markkanen,...,In The Paint (Non-RA),Center(C),Less Than 8 ft.,6,-5,-61,1,1,P_HIGH,0022400006-1628374


In [25]:
# se deja por el momento únicamente la temporada 2025-26, debido a que los otros archivos con tienen solo está temporada
df_shots = df_shots[df_shots['season_num'] == '2025-26']

# Clustering

Con el fin de conocer el perfil ofensivo de cada jugador y descubrir jugadores con perfiles similares, se realizará en clustering.

# Limpieza
- Se eliminarán los tiros desde la zona **Backcourt** ya que estos tiros son más circunstanciales que de un tipo de juego com tal.
- Se eliminarán los tiros incongruentes, como por ejemplo tiros de 3 pts marcados como mid range.
- Se agruparan las zonas de las esquinas ya que estás zonas no tienen tanto volumnen y la diferencia de lados no define un perfil como tal.

In [12]:
df_shots['shot_zone_basic'].value_counts()

shot_zone_basic
Above the Break 3        71637
Restricted Area          66364
In The Paint (Non-RA)    46750
Mid-Range                23702
Left Corner 3            13024
Right Corner 3           12122
Backcourt                   34
Name: count, dtype: int64

In [13]:
# ELiminar backcourt
df_shots = df_shots[df_shots['shot_zone_basic'] != 'Backcourt']
df_shots['shot_zone_basic'].value_counts()

shot_zone_basic
Above the Break 3        71637
Restricted Area          66364
In The Paint (Non-RA)    46750
Mid-Range                23702
Left Corner 3            13024
Right Corner 3           12122
Name: count, dtype: int64

In [14]:
df_shots[['shot_type', 'shot_zone_basic']].value_counts()

shot_type       shot_zone_basic      
3PT Field Goal  Above the Break 3        71625
2PT Field Goal  Restricted Area          66364
                In The Paint (Non-RA)    46750
                Mid-Range                23694
3PT Field Goal  Left Corner 3            13024
                Right Corner 3           12122
2PT Field Goal  Above the Break 3           12
3PT Field Goal  Mid-Range                    8
Name: count, dtype: int64

Se evidencia una incongruencia en **11** tiros clasificados como de **3Pts** pero en la zona de **Mid-Range**, lo cual no es posible. Lo mismo ocurrre con **22** tiros de **2Pts**.

In [15]:
# Eliminar tiros incongruentes
df_shots = df_shots[~((df_shots['shot_type'] == '3PT Field Goal') & (df_shots['shot_zone_basic'] == 'Mid-Range'))]
df_shots = df_shots[~((df_shots['shot_type'] == '2PT Field Goal') & (df_shots['shot_zone_basic'] == 'Above the Break 3'))]
df_shots[['shot_type', 'shot_zone_basic']].value_counts()

shot_type       shot_zone_basic      
3PT Field Goal  Above the Break 3        71625
2PT Field Goal  Restricted Area          66364
                In The Paint (Non-RA)    46750
                Mid-Range                23694
3PT Field Goal  Left Corner 3            13024
                Right Corner 3           12122
Name: count, dtype: int64

In [16]:
df_shots['shot_zone_basic'] = np.where((df_shots['shot_zone_basic'] == 'Left Corner 3') | (df_shots['shot_zone_basic'] == 'Right Corner 3'),
                    'Corner 3', df_shots['shot_zone_basic'])
df_shots[['shot_type', 'shot_zone_basic']].value_counts()

shot_type       shot_zone_basic      
3PT Field Goal  Above the Break 3        71625
2PT Field Goal  Restricted Area          66364
                In The Paint (Non-RA)    46750
3PT Field Goal  Corner 3                 25146
2PT Field Goal  Mid-Range                23694
Name: count, dtype: int64

# Criterio de Inclusión
Con el fin de no incluir jugadores de rotación corta que no tengan suficientes tiros para definir un perfil solido, se limita a jugadores que superen los 150 tiros por temporada.

In [17]:
jugadores_excluidos = df_shots.groupby('player_id')['shot_attempted_flag'].sum()
jugadores_excluidos = (jugadores_excluidos[jugadores_excluidos.values < 150]
                       .sort_values(ascending=False).index.to_list())
print(len(jugadores_excluidos))
jugadores_excluidos[0:5]

195


[1626166, 1642277, 1629680, 1627827, 1630623]

In [18]:
print(len(df_shots['player_id'].unique()))
df_shots = df_shots[~(df_shots['player_id'].isin(jugadores_excluidos))]
print(len(df_shots['player_id'].unique()))

582
387


In [19]:
df_shots_aggregated = df_shots.groupby(['player_id', 'season_num', 'shot_zone_basic']).agg(
    shot_attempted = ('shot_attempted_flag', 'sum'),
    shot_made = ('shot_made_flag', 'sum')
).reset_index()

df_shots_aggregated.head(10)

,player_id,season_num,shot_zone_basic,shot_attempted,shot_made
0,2544,2025-26,Above the Break 3,238,78
1,2544,2025-26,Corner 3,56,16
2,2544,2025-26,In The Paint (Non-RA),227,88
3,2544,2025-26,Mid-Range,186,80
4,2544,2025-26,Restricted Area,394,295
5,201142,2025-26,Above the Break 3,405,163
6,201142,2025-26,Corner 3,49,24
7,201142,2025-26,In The Paint (Non-RA),383,234
8,201142,2025-26,Mid-Range,423,204
9,201142,2025-26,Restricted Area,128,98


In [20]:
df_shots_aggregated['attempted_pct'] = df_shots_aggregated['shot_attempted']/df_shots_aggregated.groupby(['player_id', 'season_num'])['shot_attempted'].transform('sum')
df_shots_aggregated.head(10)

,player_id,season_num,shot_zone_basic,shot_attempted,shot_made,attempted_pct
0,2544,2025-26,Above the Break 3,238,78,0.216167
1,2544,2025-26,Corner 3,56,16,0.050863
2,2544,2025-26,In The Paint (Non-RA),227,88,0.206176
3,2544,2025-26,Mid-Range,186,80,0.168937
4,2544,2025-26,Restricted Area,394,295,0.357856
5,201142,2025-26,Above the Break 3,405,163,0.291787
6,201142,2025-26,Corner 3,49,24,0.035303
7,201142,2025-26,In The Paint (Non-RA),383,234,0.275937
8,201142,2025-26,Mid-Range,423,204,0.304755
9,201142,2025-26,Restricted Area,128,98,0.092219


In [21]:
df_shots_pv = df_shots_aggregated.pivot_table(values="attempted_pct", 
                                              index=['player_id', 'season_num'], 
                                              columns=['shot_zone_basic'], fill_value= 0).reset_index().rename_axis(columns=None)
df_shots_pv.head()

,player_id,season_num,Above the Break 3,Corner 3,In The Paint (Non-RA),Mid-Range,Restricted Area
0,2544,2025-26,0.216167,0.050863,0.206176,0.168937,0.357856
1,201142,2025-26,0.291787,0.035303,0.275937,0.304755,0.092219
2,201143,2025-26,0.484568,0.148148,0.175926,0.049383,0.141975
3,201144,2025-26,0.537549,0.221344,0.126482,0.043478,0.071146
4,201566,2025-26,0.323004,0.128725,0.209774,0.138260,0.200238


In [22]:
print(df_shots_pv.isnull().sum())

player_id                0
season_num               0
Above the Break 3        0
Corner 3                 0
In The Paint (Non-RA)    0
Mid-Range                0
Restricted Area          0
dtype: int64


# Datos de rol

Para incluir más información ofensiva de cada jugador se agregaran las siguientes variables por cada 36 minutos.

- assists
- rebounds_offensive
- field_goals_attempted
- three_pointers_attempted
- free_throws_attempted
- turnovers

In [23]:
df_boxscore = pd.read_parquet(RAIZ/"src/data/parquet/fct_boxscore.parquet")
df_boxscore = df_boxscore[[
    'game_id','person_id', 'minutes', 'assists', 'rebounds_offensive', 'field_goals_attempted', 
    'three_pointers_attempted', 'free_throws_attempted', 'turnovers']]
df_boxscore.head()

,game_id,person_id,minutes,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers
0,0022500001,1629652,45:15,5,2,12,8,2,1
1,0022500001,1631096,38:30,2,2,17,5,4,3
2,0022500001,1628392,36:51,5,5,5,1,2,1
3,0022500001,1641717,41:39,5,1,10,8,2,2
4,0022500001,1628983,47:13,5,0,26,9,14,3


In [24]:
print(df_boxscore['minutes'][df_boxscore['minutes'] == ''].count())
df_boxscore = df_boxscore[~(df_boxscore['minutes'] == '')]
print(df_boxscore['minutes'][df_boxscore['minutes'] == ''].count())

6015
0


In [25]:
def extraer_segundos(minutes):
    resultado = float(minutes.split(":")[0]) + float(minutes.split(":")[1])/60
    return resultado

df_boxscore['minutes'] = df_boxscore['minutes'].loc[:].apply(extraer_segundos)
df_boxscore.head()

,game_id,person_id,minutes,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers
0,0022500001,1629652,45.250000,5,2,12,8,2,1
1,0022500001,1631096,38.500000,2,2,17,5,4,3
2,0022500001,1628392,36.850000,5,5,5,1,2,1
3,0022500001,1641717,41.650000,5,1,10,8,2,2
4,0022500001,1628983,47.216667,5,0,26,9,14,3


In [26]:
df_games = pd.read_parquet(RAIZ/"src/data/parquet/fct_games.parquet")
df_games = df_games[['game_id', 'season_num']]
print(df_games.shape)
df_games.drop_duplicates(inplace=True)
print(df_games.shape)
df_games.head()

(2630, 2)
(1315, 2)


,game_id,season_num
0,0022501188,2025-26
1,0022501200,2025-26
2,0022501187,2025-26
3,0022501199,2025-26
4,0022501198,2025-26


In [27]:
print(df_boxscore.shape)
df_boxscore = df_boxscore.merge(df_games, how= 'left', on= 'game_id')
print(df_boxscore.shape)
df_boxscore.head()

(28572, 9)
(28572, 10)


,game_id,person_id,minutes,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers,season_num
0,0022500001,1629652,45.250000,5,2,12,8,2,1,2025-26
1,0022500001,1631096,38.500000,2,2,17,5,4,3,2025-26
2,0022500001,1628392,36.850000,5,5,5,1,2,1,2025-26
3,0022500001,1641717,41.650000,5,1,10,8,2,2,2025-26
4,0022500001,1628983,47.216667,5,0,26,9,14,3,2025-26


In [28]:
df_boxscore_aggregated = df_boxscore.groupby(['person_id', 'season_num']).agg(
    minutes = ('minutes', 'sum'),
    minutes_prom = ('minutes', 'mean'),
    assists = ('assists', 'sum'),
    rebounds_offensive = ('rebounds_offensive', 'sum'), 
    field_goals_attempted = ('field_goals_attempted', 'sum'),
    three_pointers_attempted = ('three_pointers_attempted', 'sum'),
    free_throws_attempted = ('free_throws_attempted', 'sum'),
    turnovers = ('turnovers', 'sum')
).reset_index()
df_boxscore_aggregated.head()

,person_id,season_num,minutes,minutes_prom,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers
0,2544,2025-26,2372.866667,33.898095,505,47,1102,295,379,217
1,101108,2025-26,227.533333,14.220833,53,6,56,30,2,16
2,200768,2025-26,120.816667,7.551042,11,0,25,25,6,4
3,201142,2025-26,2880.816667,36.466034,376,43,1388,454,476,255
4,201143,2025-26,969.200000,21.537778,116,44,324,205,26,42


In [29]:
columnas = df_boxscore_aggregated.columns.to_list()[4:]
columnas

['assists',
 'rebounds_offensive',
 'field_goals_attempted',
 'three_pointers_attempted',
 'free_throws_attempted',
 'turnovers']

In [30]:
df_boxscore_aggregated[columnas] = df_boxscore_aggregated[columnas].div(df_boxscore_aggregated['minutes'], axis=0)*36
df_boxscore_aggregated.rename(columns={'person_id' : 'player_id'}, inplace= True)
df_boxscore_aggregated.head()

,player_id,season_num,minutes,minutes_prom,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers
0,2544,2025-26,2372.866667,33.898095,7.661619,0.713062,16.719018,4.475599,5.750007,3.292220
1,101108,2025-26,227.533333,14.220833,8.385585,0.949311,8.860240,4.746557,0.316437,2.531497
2,200768,2025-26,120.816667,7.551042,3.277693,0.000000,7.449303,7.449303,1.787833,1.191889
3,201142,2025-26,2880.816667,36.466034,4.698668,0.537348,17.345082,5.673391,5.948313,3.186596
4,201143,2025-26,969.200000,21.537778,4.308708,1.634338,12.034668,7.614527,0.965745,1.560050


In [31]:
df_completo = df_shots_pv.merge(df_boxscore_aggregated, how= 'left', on=['player_id', 'season_num'])
df_completo.head()

,player_id,season_num,Above the Break 3,Corner 3,In The Paint (Non-RA),Mid-Range,Restricted Area,minutes,minutes_prom,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers
0,2544,2025-26,0.216167,0.050863,0.206176,0.168937,0.357856,2372.866667,33.898095,7.661619,0.713062,16.719018,4.475599,5.750007,3.292220
1,201142,2025-26,0.291787,0.035303,0.275937,0.304755,0.092219,2880.816667,36.466034,4.698668,0.537348,17.345082,5.673391,5.948313,3.186596
2,201143,2025-26,0.484568,0.148148,0.175926,0.049383,0.141975,969.200000,21.537778,4.308708,1.634338,12.034668,7.614527,0.965745,1.560050
3,201144,2025-26,0.537549,0.221344,0.126482,0.043478,0.071146,1159.150000,17.562879,5.869818,0.590088,7.857482,5.962990,1.614976,1.211232
4,201566,2025-26,0.323004,0.128725,0.209774,0.138260,0.200238,1852.583333,28.946615,8.278170,1.554586,16.303720,7.364851,3.556115,4.061356


In [32]:
df_completo.drop(columns=['minutes', 'minutes_prom'], inplace=True)
df_completo.head()

,player_id,season_num,Above the Break 3,Corner 3,In The Paint (Non-RA),Mid-Range,Restricted Area,assists,rebounds_offensive,field_goals_attempted,three_pointers_attempted,free_throws_attempted,turnovers
0,2544,2025-26,0.216167,0.050863,0.206176,0.168937,0.357856,7.661619,0.713062,16.719018,4.475599,5.750007,3.292220
1,201142,2025-26,0.291787,0.035303,0.275937,0.304755,0.092219,4.698668,0.537348,17.345082,5.673391,5.948313,3.186596
2,201143,2025-26,0.484568,0.148148,0.175926,0.049383,0.141975,4.308708,1.634338,12.034668,7.614527,0.965745,1.560050
3,201144,2025-26,0.537549,0.221344,0.126482,0.043478,0.071146,5.869818,0.590088,7.857482,5.962990,1.614976,1.211232
4,201566,2025-26,0.323004,0.128725,0.209774,0.138260,0.200238,8.278170,1.554586,16.303720,7.364851,3.556115,4.061356
